In [1]:
import random
import numpy as np
import torch

import config
from dataset import (
    load_dataframe,
    split_data,
    get_class_weights,
    get_sampler,
    get_loaders_m1,
    get_loaders_m2,
    load_test_df,
    get_test_loaders
)

from model import (
    build_model,
    EnsembleModel,
    load_model,
    get_criterion,
    count_parameters
)

from train import train_model

from evaluate import (
    run_inference,
    run_ensemble_inference,
    print_report
)

from plots import (
    TrainingHistory,
    plot_loss,
    plot_f1,
    plot_confusion_matrix,
    plot_all_confusion_matrices,
    plot_roc_auc,
    plot_roc_auc_ensemble
)

c:\Users\danish\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TRAIN_MODEL1 = False
TRAIN_MODEL2 = False

random.seed(config.SEED)
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [3]:
# Load data
df, classes, class_to_idx = load_dataframe()

# Split data
df_train, df_val = split_data(df)

# Calculate class weights
class_weights = get_class_weights(df_train)

# Get unhealthy class index
unhealthy_idx = class_to_idx["unhealthy"]

print(df_train.head())
print(classes)
print(class_weights)

Classes: ['healthy', 'rubbish', 'unhealthy']
Label distribution:
label
rubbish      50371
healthy      28895
unhealthy     2366
Name: count, dtype: int64
Train: 69387 | Val: 12245
                                image_name    label  target
44775  isbi2025_ps3c_train_image_11825.png  healthy       0
20296  isbi2025_ps3c_train_image_03475.png  healthy       0
64544  isbi2025_ps3c_train_image_04756.png  healthy       0
51237  isbi2025_ps3c_train_image_23741.png  healthy       0
40787  isbi2025_ps3c_train_image_15487.png  healthy       0
['healthy', 'rubbish', 'unhealthy']
[0.2175966  0.12482518 2.6575782 ]


In [4]:
history1 = TrainingHistory("Model 1")

sampler_m1 = get_sampler(df_train, class_weights)

train_loader_m1, val_loader_m1 = get_loaders_m1(
    df_train,
    df_val,
    sampler_m1
)

model1 = build_model(len(classes)).to(device)

criterion1 = get_criterion(class_weights, device)

count_parameters(model1)

c:\Users\danish\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\sampler.py:264: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights_tensor = torch.as_tensor(weights, dtype=torch.double)


Total parameters:       20,181,331  (20.18M)
Trainable parameters:   20,181,331  (20.18M)


(20181331, 20181331)

In [5]:
import os

print(config.MODEL1_BEST)
print(os.path.exists(config.MODEL1_BEST))

C:\Users\danish\Desktop\pap_cell_project\data\best_model_3.pth
True


In [6]:
if TRAIN_MODEL1:
    train_model(
        model1, train_loader_m1, val_loader_m1, criterion1,
        config.MODEL1_CKPT, config.MODEL1_BEST, device,
        model_name='Model1', history=history1  
    )

In [7]:
plot_loss(history1)
plot_f1(history1)

Saved: c:\Users\danish\Desktop\pap_cell_project\results\loss_curves.png
Saved: c:\Users\danish\Desktop\pap_cell_project\results\f1_curves.png


In [8]:
history2 = TrainingHistory("Model 2")

sampler_m2 = get_sampler(df_train, class_weights)

train_loader_m2, val_loader_m2 = get_loaders_m2(
    df_train,
    df_val,
    sampler_m2,
    unhealthy_idx
)

model2 = build_model(len(classes)).to(device)

criterion2 = get_criterion(class_weights, device)

count_parameters(model2)

Total parameters:       20,181,331  (20.18M)
Trainable parameters:   20,181,331  (20.18M)


(20181331, 20181331)

In [9]:
if TRAIN_MODEL2:
    train_model(
        model2, train_loader_m2, val_loader_m2, criterion2,
        config.MODEL2_CKPT, config.MODEL2_BEST, device,
        model_name='Model2', history=history2,   
    )

In [10]:
plot_loss(history1, history2)

plot_f1(history1, history2)

Saved: c:\Users\danish\Desktop\pap_cell_project\results\loss_curves.png
Saved: c:\Users\danish\Desktop\pap_cell_project\results\f1_curves.png


In [11]:
model1 = load_model(
    config.MODEL1_BEST,
    len(classes),
    device
)

model2 = load_model(
    config.MODEL2_BEST,
    len(classes),
    device
)

Model loaded from: C:\Users\danish\Desktop\pap_cell_project\data\best_model_3.pth
Model loaded from: C:\Users\danish\Desktop\pap_cell_project\data\best_model_v2_6.pth


In [12]:
df_test = load_test_df(class_to_idx)

test_loader_m1, test_loader_m2, \
test_ds_m1, test_ds_m2 = get_test_loaders(df_test)

Found test CSV: C:\Users\danish\Desktop\pap_cell_project\data\isbi2025-ps3c-test-dataset-annotated.csv
Test samples: 18159
label
rubbish      11757
healthy       5826
unhealthy      576
Name: count, dtype: int64


In [13]:
y_true_m1, y_pred_m1 = run_inference(
    model1,
    test_ds_m1,
    device
)

print_report(
    y_true_m1,
    y_pred_m1,
    classes,
    title="Model 1"
)

plot_confusion_matrix(
    y_true_m1,
    y_pred_m1,
    classes,
    title="Model 1",
    filename="confusion_matrix_model1.png"
)


Model 1
              precision    recall  f1-score   support

     healthy     0.7582    0.8514    0.8021      5826
     rubbish     0.9140    0.8654    0.8891     11757
   unhealthy     0.4515    0.3802    0.4128       576

    accuracy                         0.8455     18159
   macro avg     0.7079    0.6990    0.7013     18159
weighted avg     0.8494    0.8455    0.8461     18159

Confusion Matrix:
[[ 4960   780    86]
 [ 1402 10175   180]
 [  180   177   219]]

Macro-F1:    0.7013
Weighted-F1: 0.8461
Accuracy:    0.8455
Saved: c:\Users\danish\Desktop\pap_cell_project\results\confusion_matrix_model1.png


In [14]:
y_true_m2, y_pred_m2 = run_inference(
    model2,
    test_ds_m2,
    device
)

print_report(
    y_true_m2,
    y_pred_m2,
    classes,
    title="Model 2"
)

plot_confusion_matrix(
    y_true_m2,
    y_pred_m2,
    classes,
    title="Model 2",
    filename="confusion_matrix_model2.png"
)


Model 2
              precision    recall  f1-score   support

     healthy     0.7420    0.8785    0.8045      5826
     rubbish     0.9315    0.8456    0.8865     11757
   unhealthy     0.3895    0.3976    0.3935       576

    accuracy                         0.8420     18159
   macro avg     0.6876    0.7072    0.6948     18159
weighted avg     0.8535    0.8420    0.8445     18159

Confusion Matrix:
[[5118  584  124]
 [1580 9942  235]
 [ 200  147  229]]

Macro-F1:    0.6948
Weighted-F1: 0.8445
Accuracy:    0.8420
Saved: c:\Users\danish\Desktop\pap_cell_project\results\confusion_matrix_model2.png


In [21]:
ensemble = EnsembleModel(
    model1,
    model2
).to(device)

y_true, y_pred = run_ensemble_inference(
    ensemble,
    test_ds_m1,
    test_ds_m2,
    device
)

print_report(
    y_true,
    y_pred,
    classes,
    title="Ensemble"
)


Ensemble
              precision    recall  f1-score   support

     healthy     0.7599    0.8714    0.8119      5826
     rubbish     0.9292    0.8579    0.8921     11757
   unhealthy     0.4038    0.4375    0.4200       576

    accuracy                         0.8489     18159
   macro avg     0.6977    0.7223    0.7080     18159
weighted avg     0.8583    0.8489    0.8514     18159

Confusion Matrix:
[[ 5077   622   127]
 [ 1426 10086   245]
 [  178   146   252]]

Macro-F1:    0.7080
Weighted-F1: 0.8514
Accuracy:    0.8489


In [22]:
plot_confusion_matrix(
    y_true,
    y_pred,
    classes,
    title="Ensemble",
    filename="confusion_matrix_ensemble.png"
)

plot_all_confusion_matrices(
    y_true_m1,
    y_pred_m1,
    y_true_m2,
    y_pred_m2,
    y_true,
    y_pred,
    classes
)

Saved: c:\Users\danish\Desktop\pap_cell_project\results\confusion_matrix_ensemble.png
Saved: c:\Users\danish\Desktop\pap_cell_project\results\all_confusion_matrices.png


In [23]:
plot_roc_auc(
    model1,
    test_ds_m1,
    classes,
    device,
    title="Model 1 ROC",
    filename="roc_auc_model1.png"
)

plot_roc_auc(
    model2,
    test_ds_m2,
    classes,
    device,
    title="Model 2 ROC",
    filename="roc_auc_model2.png"
)

plot_roc_auc_ensemble(
    ensemble,
    test_ds_m1,
    test_ds_m2,
    classes,
    device,
    title="Ensemble ROC",
    filename="roc_auc_ensemble.png"
)

Saved: c:\Users\danish\Desktop\pap_cell_project\results\roc_auc_model1.png
AUC scores: {'healthy': 0.9134235445281191, 'rubbish': 0.9121115825711201, 'unhealthy': 0.6685046260750598}
Macro-average AUC: 0.8313
Saved: c:\Users\danish\Desktop\pap_cell_project\results\roc_auc_model2.png
AUC scores: {'healthy': 0.9255986168134529, 'rubbish': 0.9278783433358159, 'unhealthy': 0.6904076874285137}
Macro-average AUC: 0.8480
Saved: c:\Users\danish\Desktop\pap_cell_project\results\roc_auc_ensemble.png
Ensemble AUC scores: {'healthy': 0.9333526257522088, 'rubbish': 0.9343586798024998, 'unhealthy': 0.7113788097088729}
Ensemble Macro-average AUC: 0.8597


({'healthy': 0.9333526257522088,
  'rubbish': 0.9343586798024998,
  'unhealthy': 0.7113788097088729},
 np.float64(0.8596967050878606))

In [24]:
import inspect
import plots

print(inspect.getsource(plots.plot_f1))

def plot_f1(history1, history2=None):
    """
    Plot macro-F1 and weighted-F1 per epoch for one or two models.
    """
    fig, axes = plt.subplots(
        1, 2 if history2 else 1,
        figsize=(14 if history2 else 7, 5)
    )
    if history2 is None:
        axes = [axes]

    for ax, history in zip(axes, [h for h in [history1, history2] if h]):
        epochs = range(1, len(history.val_macro_f1) + 1)
        ax.plot(
        epochs,
        history.val_macro_f1,
        marker='o',
        linestyle='-',
        color='green',
        markersize=4,
        label='Val Macro-F1'
        )

        ax.plot(
            epochs,
            history.val_weighted_f1,
            marker='p',
            linestyle='-',
            color='purple',
            markersize=4,
            label='Val Weighted-F1'
        )

        ax.plot(
            epochs,
            history.val_acc,
            marker='o',
            linestyle='--',
            color='blue',
            markersize=3,
 

In [25]:
import inspect
import plots

print(inspect.getsource(plots.plot_f1))


def plot_f1(history1, history2=None):
    """
    Plot macro-F1 and weighted-F1 per epoch for one or two models.
    """
    fig, axes = plt.subplots(
        1, 2 if history2 else 1,
        figsize=(14 if history2 else 7, 5)
    )
    if history2 is None:
        axes = [axes]

    for ax, history in zip(axes, [h for h in [history1, history2] if h]):
        epochs = range(1, len(history.val_macro_f1) + 1)
        ax.plot(
        epochs,
        history.val_macro_f1,
        marker='o',
        linestyle='-',
        color='green',
        markersize=4,
        label='Val Macro-F1'
        )

        ax.plot(
            epochs,
            history.val_weighted_f1,
            marker='p',
            linestyle='-',
            color='purple',
            markersize=4,
            label='Val Weighted-F1'
        )

        ax.plot(
            epochs,
            history.val_acc,
            marker='o',
            linestyle='--',
            color='blue',
            markersize=3,
 